<a href="https://colab.research.google.com/github/akriti404/Deep-Learning-Lab/blob/main/Lab_Assessment_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

# 1. Normalize features using Standard Scaling

data = load_breast_cancer()

X = data.data[:, :10]

radius = data.data[:, 0]
texture = data.data[:, 1]

severity_score = radius + texture

q1 = torch.tensor(severity_score).quantile(1 / 3).item()
q2 = torch.tensor(severity_score).quantile(2 / 3).item()

y = torch.tensor(
    [0 if s <= q1 else 1 if s <= q2 else 2 for s in severity_score],
    dtype=torch.long
)

scaler = StandardScaler()
X = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y.numpy(),
    test_size=0.2,
    random_state=42,
    stratify=y.numpy()
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)


# 2. Implement Custom Penalty Cross-Entropy Loss

class CustomPenaltyLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.ce = nn.CrossEntropyLoss(reduction='none')

    def forward(self, outputs, targets):
        ce_loss = self.ce(outputs, targets)

        predictions = torch.argmax(outputs, dim=1)

        penalty = torch.where(
            (targets == 2) & (predictions == 0),
            torch.tensor(3.0, device=outputs.device),
            torch.tensor(0.0, device=outputs.device)
        )

        custom_loss = ce_loss + penalty

        return custom_loss.mean()


# 3. Train two architectures

class ShallowSoftmax(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 3),
            nn.Softmax(dim=1)
        )

    def forward(self, x):
        return self.network(x)


class DeepNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(10, 32),
            nn.LeakyReLU(0.1),
            nn.Linear(32, 16),
            nn.LeakyReLU(0.1),
            nn.Linear(16, 3)
        )

    def forward(self, x):
        return self.network(x)


# 4. Run both models for 40 epochs using Adam

def train(model, X_train, y_train, epochs=40, lr=0.005):
    criterion = CustomPenaltyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    losses = []

    for epoch in range(1, epochs + 1):
        model.train()

        optimizer.zero_grad()

        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        print(
            f"Epoch {epoch:02d} | "
            f"Training Loss: {loss.item():.4f}"
        )

    return losses


torch.manual_seed(42)

model_a = ShallowSoftmax()
loss_a = train(
    model_a,
    X_train,
    y_train,
    epochs=40,
    lr=0.005
)

print("\nTraining Model B")

torch.manual_seed(42)

model_b = DeepNetwork()
loss_b = train(
    model_b,
    X_train,
    y_train,
    epochs=40,
    lr=0.005
)


# 5. Output required results

first_better_epoch = None

for epoch in range(40):
    if loss_b[epoch] < loss_a[epoch]:
        first_better_epoch = epoch + 1
        break

print("\nFinal Results")

if first_better_epoch is not None:
    print(
        f"Model B first drops below Model A at Epoch: "
        f"{first_better_epoch}"
    )
else:
    print("Model B never drops below Model A.")

print(f"Model A Final Custom Loss: {loss_a[-1]:.4f}")
print(f"Model B Final Custom Loss: {loss_b[-1]:.4f}")

Epoch 01 | Training Loss: 1.9338
Epoch 02 | Training Loss: 1.9416
Epoch 03 | Training Loss: 1.9559
Epoch 04 | Training Loss: 1.9568
Epoch 05 | Training Loss: 1.9577
Epoch 06 | Training Loss: 1.9518
Epoch 07 | Training Loss: 1.9524
Epoch 08 | Training Loss: 1.9462
Epoch 09 | Training Loss: 1.9531
Epoch 10 | Training Loss: 1.9468
Epoch 11 | Training Loss: 1.9468
Epoch 12 | Training Loss: 1.9468
Epoch 13 | Training Loss: 1.9400
Epoch 14 | Training Loss: 1.9199
Epoch 15 | Training Loss: 1.9128
Epoch 16 | Training Loss: 1.9056
Epoch 17 | Training Loss: 1.8851
Epoch 18 | Training Loss: 1.8316
Epoch 19 | Training Loss: 1.7910
Epoch 20 | Training Loss: 1.7043
Epoch 21 | Training Loss: 1.6505
Epoch 22 | Training Loss: 1.6098
Epoch 23 | Training Loss: 1.5165
Epoch 24 | Training Loss: 1.4892
Epoch 25 | Training Loss: 1.4422
Epoch 26 | Training Loss: 1.4020
Epoch 27 | Training Loss: 1.3621
Epoch 28 | Training Loss: 1.3290
Epoch 29 | Training Loss: 1.3093
Epoch 30 | Training Loss: 1.2834
Epoch 31 |